# KD-BERT Paper-Faithful Corrected Baseline

This notebook implements the paper method with corrected code: 12-layer teacher, MPM + SPP, attention and hidden-state distillation, then a 2-layer student with hidden size 128. It fixes transport-header removal, padding-aware distillation, best-checkpoint selection, and window overlap across splits. The model and pretraining objectives follow the paper; capture-level splitting is stricter than the paper description where enough captures exist.

Run the baseline first, then the flow-aware notebook, with identical dataset path, experiment root, seed, and QUICK_RUN setting. The baseline creates the shared teacher checkpoint; the second notebook reuses it. QUICK_RUN defaults to False and the full 500,000-step teacher run is compute intensive. No results are claimed until the cells finish.

Input layout: one folder per class, with PCAP/PCAPNG captures inside. When a class has fewer than three captures, the notebook uses a chronological per-capture fallback and records that limitation in its output.


## 1. Dependencies and Drive


In [ ]:
!pip -q install scapy scikit-learn matplotlib


In [ ]:
# Mount Drive in Colab. For a local runtime, set DATASET_PATH and EXPERIMENT_ROOT below.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Google Drive mount skipped; using configured local paths.')


Mounted at /content/drive


## 2. Shared experiment settings


In [ ]:
import os, json, math, time, random, hashlib, pickle, ipaddress, warnings
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Identical experiment settings in both notebooks
DATASET_PATH = '/content/drive/MyDrive/CNS-Pro/dataset-100'
DATASET_NAME = 'dataset-100'
EXPERIMENT_ROOT = '/content/drive/MyDrive/KD_BERT_Comparison'
RANDOM_SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.80, 0.10, 0.10
SAMPLES_PER_CLASS = 5000
WINDOW_SIZE, WINDOW_STRIDE = 5, 1
FLOW_CONTEXT_SECONDS = 1.0
MAX_SEQ_LEN, MAX_PACKET_TOKENS, PAYLOAD_MAX_BYTES = 256, 64, 80
VOCAB_SIZE = 65540
PAD_ID, CLS_ID, SEP_ID, MASK_ID, TOKEN_OFFSET = 0, 1, 2, 3, 4

TEACHER_LAYERS, TEACHER_HIDDEN, TEACHER_HEADS, TEACHER_FFN = 12, 768, 12, 3072
STUDENT_LAYERS, STUDENT_HIDDEN, STUDENT_HEADS, STUDENT_FFN = 2, 128, 2, 512
PRETRAIN_STEPS = 500_000
PRETRAIN_LR, DISTILL_LR, FINETUNE_LR = 2e-5, 2e-5, 2e-5
PRETRAIN_BATCH_SIZE = DISTILL_BATCH_SIZE = FINETUNE_BATCH_SIZE = 32
MICRO_BATCH_SIZE, GRAD_ACCUM_STEPS = 8, 4
DISTILL_EPOCHS, FINETUNE_EPOCHS = 10, 10
WARMUP_RATIO, MASK_PROB, DOS_AUX_WEIGHT = 0.10, 0.15, 0.20
CHECKPOINT_EVERY = 10_000
QUICK_RUN = True  # Set True in BOTH notebooks for a smoke run only, never for reported results.
if QUICK_RUN:
    PRETRAIN_STEPS, DISTILL_EPOCHS, FINETUNE_EPOCHS = 1_000, 1, 1
    SAMPLES_PER_CLASS = 200

MODEL_VARIANT = 'paper_baseline'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs(EXPERIMENT_ROOT, exist_ok=True)
DATA_SHARED_DIR = os.path.join(EXPERIMENT_ROOT, 'shared', DATASET_NAME)
os.makedirs(DATA_SHARED_DIR, exist_ok=True)
SHARED_DIR = os.path.join(DATA_SHARED_DIR, f'seed_{RANDOM_SEED}')
os.makedirs(SHARED_DIR, exist_ok=True)
SHARED_TEACHER_PATH = os.path.join(SHARED_DIR, f'teacher_12x768_{DATASET_NAME}.pt')
PACKET_CACHE_PATH = os.path.join(DATA_SHARED_DIR, 'packet_records.pkl')
SPLIT_MANIFEST_PATH = os.path.join(SHARED_DIR, 'split_manifest.json')
VARIANT_DIR = os.path.join(EXPERIMENT_ROOT, DATASET_NAME, f'seed_{RANDOM_SEED}', MODEL_VARIANT)
os.makedirs(VARIANT_DIR, exist_ok=True)
RESULTS_PATH = os.path.join(VARIANT_DIR, 'metrics.json')
BEST_MODEL_PATH = os.path.join(VARIANT_DIR, 'best_student.pt')

def seed_everything(seed=RANDOM_SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f'Device: {DEVICE} | Dataset: {DATASET_PATH} | Variant: {MODEL_VARIANT}')
print(f'Effective batch: {MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS} | Pretraining steps: {PRETRAIN_STEPS:,}')


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


Device: cpu | Dataset: /content/drive/MyDrive/CNS-Pro/dataset-100 | Variant: paper_baseline
Effective batch: 32 | Pretraining steps: 1,000


## 3. PCAP preprocessing and shared capture split


In [ ]:
from scapy.all import PcapReader, IP, IPv6, TCP, UDP, Raw, NoPayload
warnings.filterwarnings('ignore')

def dataset_file_signature(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'Dataset folder not found: {root}')
    files = sorted([p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in {'.pcap', '.pcapng'}])
    if not files:
        raise FileNotFoundError(f'No PCAP/PCAPNG files found below {root}')
    rows = [(str(p.relative_to(root)), p.stat().st_size, int(p.stat().st_mtime)) for p in files]
    return files, hashlib.sha256(json.dumps(rows, sort_keys=True).encode()).hexdigest()

def _anon_ip(address, capture_id, packet_index, stable):
    version = ipaddress.ip_address(address).version
    suffix = 'stable' if stable else f'packet-{packet_index}'
    digest = hashlib.sha256(f'{RANDOM_SEED}|{capture_id}|{address}|{suffix}'.encode()).digest()
    if version == 4:
        return str(ipaddress.IPv4Address(0x0A000000 | int.from_bytes(digest[:3], 'big')))
    return str(ipaddress.IPv6Address(int.from_bytes(digest[:16], 'big')))

def _stable_endpoint_id(address, capture_id):
    return hashlib.sha256(f'{RANDOM_SEED}|{capture_id}|{address}'.encode()).hexdigest()[:20]

def _port_bucket(port):
    if not port: return 0
    if port < 1024: return 1
    if port < 49152: return 2
    return 3

def _packet_views(pkt, capture_id, packet_index):
    # Strip Ethernet but keep network and transport headers. Remove only the payload layer.
    if pkt.haslayer(IP):
        l3, version = pkt[IP].copy(), 4
        src, dst, proto = str(pkt[IP].src), str(pkt[IP].dst), int(pkt[IP].proto)
    elif pkt.haslayer(IPv6):
        l3, version = pkt[IPv6].copy(), 6
        src, dst, proto = str(pkt[IPv6].src), str(pkt[IPv6].dst), int(pkt[IPv6].nh)
    else:
        return None
    raw_src, raw_dst = src, dst
    sport = int(pkt[TCP].sport if pkt.haslayer(TCP) else pkt[UDP].sport if pkt.haslayer(UDP) else 0)
    dport = int(pkt[TCP].dport if pkt.haslayer(TCP) else pkt[UDP].dport if pkt.haslayer(UDP) else 0)
    flags = int(pkt[TCP].flags) if pkt.haslayer(TCP) else 0

    raw = l3[Raw] if l3.haslayer(Raw) else None
    transport = l3[TCP] if l3.haslayer(TCP) else l3[UDP] if l3.haslayer(UDP) else None
    if raw is not None:
        payload = bytes(raw.load)
        if raw.underlayer is not None:
            raw.underlayer.remove_payload()
    elif transport is not None and not isinstance(transport.payload, NoPayload):
        payload = bytes(transport.payload)
        transport.remove_payload()
    else:
        payload = b''

    paper = l3.copy()
    paper.src, paper.dst = _anon_ip(raw_src, capture_id, packet_index, False), _anon_ip(raw_dst, capture_id, packet_index, False)
    if version == 4: paper.len, paper.chksum = None, None
    else: paper.plen = None
    if paper.haslayer(TCP):
        paper[TCP].sport, paper[TCP].dport, paper[TCP].chksum, paper[TCP].dataofs = 0, 0, None, None
    if paper.haslayer(UDP):
        paper[UDP].sport, paper[UDP].dport, paper[UDP].chksum, paper[UDP].len = 0, 0, None, None
    paper_header = bytes(paper)

    flow = l3.copy()
    flow.src, flow.dst = _anon_ip(raw_src, capture_id, packet_index, True), _anon_ip(raw_dst, capture_id, packet_index, True)
    if version == 4: flow.len, flow.chksum = None, None
    else: flow.plen = None
    if flow.haslayer(TCP):
        flow[TCP].sport, flow[TCP].dport, flow[TCP].chksum, flow[TCP].dataofs = 0, 0, None, None
    if flow.haslayer(UDP):
        flow[UDP].sport, flow[UDP].dport, flow[UDP].chksum, flow[UDP].len = 0, 0, None, None
    flow_header = bytes(flow)

    try: timestamp = float(pkt.time)
    except Exception: timestamp = float(packet_index)
    return {
        'header_paper': paper_header, 'header_flow': flow_header, 'payload': payload[:PAYLOAD_MAX_BYTES],
        'capture_id': capture_id, 'index': int(packet_index), 'timestamp': timestamp, 'label': None,
        'src_id': _stable_endpoint_id(raw_src, capture_id), 'dst_id': _stable_endpoint_id(raw_dst, capture_id),
        'protocol': proto, 'src_port_bucket': _port_bucket(sport), 'dst_port_bucket': _port_bucket(dport),
        'tcp_flags': flags, 'packet_length': int(len(pkt)),
    }

def preprocess_or_load_records():
    files, signature = dataset_file_signature(DATASET_PATH)
    meta_path = PACKET_CACHE_PATH + '.json'
    if os.path.exists(PACKET_CACHE_PATH) and os.path.exists(meta_path):
        meta = json.load(open(meta_path))
        if meta.get('dataset_signature') == signature:
            with open(PACKET_CACHE_PATH, 'rb') as f: records = pickle.load(f)
            print(f'Loaded {len(records):,} sanitized packets from shared cache.')
            return records, signature
    records = []
    class_dirs = sorted([p for p in Path(DATASET_PATH).iterdir() if p.is_dir()])
    if not class_dirs:
        raise ValueError('Expected one folder per class, with PCAP files inside.')
    for class_dir in tqdm(class_dirs, desc='Reading classes'):
        captures = sorted([p for p in class_dir.rglob('*') if p.is_file() and p.suffix.lower() in {'.pcap', '.pcapng'}])
        for path in captures:
            capture_id = str(path.relative_to(DATASET_PATH))
            try:
                with PcapReader(str(path)) as reader:
                    for index, pkt in enumerate(reader):
                        view = _packet_views(pkt, capture_id, index)
                        if view is not None:
                            view['label'] = class_dir.name
                            records.append(view)
            except Exception as exc:
                print(f'Could not read {capture_id}: {exc}')
    if not records:
        raise ValueError('No IPv4/IPv6 packets extracted. Check the PCAPs and class folders.')
    with open(PACKET_CACHE_PATH, 'wb') as f: pickle.dump(records, f, protocol=pickle.HIGHEST_PROTOCOL)
    with open(meta_path, 'w') as f:
        json.dump({'dataset_signature': signature, 'count': len(records), 'raw_addresses_saved': False}, f, indent=2)
    print(f'Parsed {len(records):,} packets. Raw IP addresses are not stored in the cache.')
    return records, signature

def make_split_manifest(records, signature):
    if os.path.exists(SPLIT_MANIFEST_PATH):
        old = json.load(open(SPLIT_MANIFEST_PATH))
        if old.get('dataset_signature') == signature and old.get('seed') == RANDOM_SEED:
            print('Reusing shared train/validation/test split.')
            return old
    by_label, by_capture = defaultdict(set), defaultdict(list)
    for r in records:
        by_label[r['label']].add(r['capture_id'])
        by_capture[r['capture_id']].append(r)
    manifest = {'dataset_signature': signature, 'seed': RANDOM_SEED, 'capture_assignment': {}, 'temporal_boundaries': {}, 'mode_by_class': {}}
    rng = random.Random(RANDOM_SEED)
    for label in sorted(by_label):
        captures = sorted(by_label[label])
        if len(captures) >= 3:
            shuffled = captures[:]; rng.shuffle(shuffled)
            nt, nv = max(1, round(len(shuffled)*TEST_RATIO)), max(1, round(len(shuffled)*VAL_RATIO))
            while nt + nv >= len(shuffled):
                nv = max(1, nv-1)
                if nt + nv >= len(shuffled): nt = max(1, nt-1)
            test_ids, val_ids = set(shuffled[:nt]), set(shuffled[nt:nt+nv])
            for cap in captures:
                manifest['capture_assignment'][cap] = 'test' if cap in test_ids else 'val' if cap in val_ids else 'train'
            manifest['mode_by_class'][label] = 'capture-level'
        else:
            manifest['mode_by_class'][label] = 'temporal-fallback'
            for cap in captures:
                n = len(by_capture[cap])
                manifest['temporal_boundaries'][cap] = [int(n*TRAIN_RATIO), int(n*(TRAIN_RATIO+VAL_RATIO))]
    with open(SPLIT_MANIFEST_PATH, 'w') as f: json.dump(manifest, f, indent=2)
    return manifest

def attach_splits(records, manifest):
    by_capture, assigned = defaultdict(list), []
    for r in records: by_capture[r['capture_id']].append(r)
    for cap, rows in by_capture.items():
        rows.sort(key=lambda x: (x['timestamp'], x['index']))
        if cap in manifest['capture_assignment']:
            for r in rows:
                item = dict(r); item['split'] = manifest['capture_assignment'][cap]; assigned.append(item)
        else:
            a, b = manifest['temporal_boundaries'][cap]
            for pos, r in enumerate(rows):
                item = dict(r); item['split'] = 'train' if pos < a else 'val' if pos < b else 'test'; assigned.append(item)
    return assigned

records, DATASET_FINGERPRINT = preprocess_or_load_records()
split_manifest = make_split_manifest(records, DATASET_FINGERPRINT)
split_records = attach_splits(records, split_manifest)
print('Split protocol by class:', split_manifest['mode_by_class'])
print('Packet counts:', {s: sum(r['split'] == s for r in split_records) for s in ('train','val','test')})


Loaded 3,764,656 sanitized packets from shared cache.
Reusing shared train/validation/test split.
Split protocol by class: {'MITM': 'temporal-fallback', 'XSS1': 'temporal-fallback', 'backdoor': 'temporal-fallback', 'ddos': 'temporal-fallback', 'dos': 'temporal-fallback', 'injection': 'temporal-fallback', 'normal': 'temporal-fallback', 'password': 'temporal-fallback', 'runsomware': 'temporal-fallback', 'scanning': 'temporal-fallback'}
Packet counts: {'train': 3011723, 'val': 376464, 'test': 376469}


## 4. Tokenization and identical window selection


In [ ]:
class PacketTokenizer:
    def __init__(self):
        self.max_seq_len, self.max_packet_tokens = MAX_SEQ_LEN, MAX_PACKET_TOKENS
    @staticmethod
    def bytes_to_ids(data):
        if len(data) % 2: data += b'\x00'
        return [((data[i] << 8) | data[i+1]) + TOKEN_OFFSET for i in range(0, len(data), 2)]
    def encode_packet_segments(self, header, payload):
        h = self.bytes_to_ids(header)
        p = self.bytes_to_ids(payload[:PAYLOAD_MAX_BYTES])
        ids = [CLS_ID] + h + [SEP_ID] + p + [SEP_ID]
        segments = [0] * (len(h) + 2) + [1] * (len(p) + 1)
        return ids, segments
    def encode_packet(self, header, payload):
        return self.encode_packet_segments(header, payload)[0]
    def pad(self, ids, length):
        ids = list(ids)
        if len(ids) > length: ids = ids[:length-1] + [SEP_ID]
        mask = [1]*len(ids)
        ids += [PAD_ID]*(length-len(ids))
        mask += [0]*(length-len(mask))
        return ids, mask
    def pad_segments(self, ids, segments, length):
        ids, segments = list(ids), list(segments)
        if len(ids) > length:
            ids, segments = ids[:length-1] + [SEP_ID], segments[:length]
        mask = [1] * len(ids)
        pad = length - len(ids)
        ids += [PAD_ID] * pad
        segments += [0] * pad
        mask += [0] * pad
        return ids, mask, segments

tokenizer = PacketTokenizer()

def flow_features(rows, context_rows=None):
    # Per-packet features describe the same five packets used by both classifiers.
    meta = []
    times = [float(r['timestamp']) for r in rows]
    prev_src = prev_dst = None
    for i, r in enumerate(rows):
        iat = max(0.0, times[i]-times[i-1]) if i else 0.0
        flags = int(r['tcp_flags'])
        size = float(r['packet_length'])
        meta.append([
            min(math.log1p(size)/12,1), min(math.log1p(iat*1000)/12,1), float(r['protocol'])/255,
            float(prev_src is not None and r['src_id'] != prev_src),
            float(prev_dst is not None and r['dst_id'] != prev_dst),
            float(bool(flags & 2)), float(bool(flags & 16)), float(bool(flags & 4)), float(bool(flags & 1)),
            float(r['dst_port_bucket'])/3,
        ])
        prev_src, prev_dst = r['src_id'], r['dst_id']

    # Victim/service context uses only packets up to the window end, within one second.
    stats = context_rows if context_rows else rows
    stats = sorted(stats, key=lambda r:(r['timestamp'],r['index']))
    stat_times = [float(r['timestamp']) for r in stats]
    elapsed = max(1e-6, FLOW_CONTEXT_SECONDS)  # rates are counts per fixed lookback window
    intervals = [max(0.0,stat_times[i]-stat_times[i-1]) for i in range(1,len(stat_times))]
    sizes = [float(r['packet_length']) for r in stats]
    syns = [float(bool(int(r['tcp_flags']) & 2)) for r in stats]
    source_counts = Counter(r['src_id'] for r in stats)
    srcs = set(source_counts)
    probs = [count / max(1, len(stats)) for count in source_counts.values()]
    source_entropy = (-sum(q * math.log(q) for q in probs) / math.log(len(probs))) if len(probs) > 1 else 0.0
    mean_iat_ms = (sum(intervals)/len(intervals)*1000) if intervals else 0.0
    context = [
        min(math.log1p(len(srcs))/5,1), source_entropy,
        min(math.log1p(len(stats)/elapsed)/20,1), min(math.log1p(sum(sizes)/elapsed)/25,1),
        sum(syns)/max(1,len(syns)), min(math.log1p(mean_iat_ms)/12,1),
        float(rows[-1]['protocol'])/255, float(rows[-1]['dst_port_bucket'])/3,
    ]
    return meta, context

def build_windows(split_records, variant):
    groups = defaultdict(list)
    for r in split_records: groups[(r['capture_id'],r['split'],r['label'])].append(r)
    labels = sorted({r['label'] for r in split_records})
    label_id = {name:i for i,name in enumerate(labels)}
    windows = []
    for (capture, split, label), rows in sorted(groups.items()):
        rows.sort(key=lambda r:(r['timestamp'],r['index']))
        for start in range(0, len(rows)-WINDOW_SIZE+1, WINDOW_STRIDE):
            w = rows[start:start+WINDOW_SIZE]
            paper_pairs = [tokenizer.encode_packet_segments(r['header_paper'],r['payload']) for r in w]
            paper_ids = [t for ids, segs in paper_pairs for t in ids]
            paper_segments = [t for ids, segs in paper_pairs for t in segs]
            teacher_ids, teacher_mask, teacher_segments = tokenizer.pad_segments(paper_ids,paper_segments,MAX_SEQ_LEN)
            teacher_ids=np.asarray(teacher_ids,dtype=np.int32)
            teacher_mask=np.asarray(teacher_mask,dtype=np.uint8)
            teacher_segments=np.asarray(teacher_segments,dtype=np.uint8)
            item = {'window_id':f'{capture}|{split}|{w[0]["index"]}','capture_id':capture,'split':split,'label':label,'label_id':label_id[label],
                    'teacher_input_ids':teacher_ids,'teacher_attention_mask':teacher_mask,'teacher_segment_ids':teacher_segments}
            if variant == 'paper_baseline':
                item['input_ids'], item['attention_mask'], item['segment_ids'] = teacher_ids, teacher_mask, teacher_segments
            else:
                flow_pairs = [tokenizer.encode_packet_segments(r['header_flow'],r['payload']) for r in w]
                padded = [tokenizer.pad_segments(ids,segs,MAX_PACKET_TOKENS) for ids,segs in flow_pairs]
                ids, masks, segments = zip(*padded)
                item['packet_ids']=np.asarray(ids,dtype=np.int32)
                item['packet_masks']=np.asarray(masks,dtype=np.uint8)
                item['packet_segments']=np.asarray(segments,dtype=np.uint8)
                end_time = float(w[-1]['timestamp'])
                end_index = int(w[-1]['index'])
                target_id, protocol, service = w[-1]['dst_id'], w[-1]['protocol'], w[-1]['dst_port_bucket']
                context_rows = [r for r in rows
                    if end_time-FLOW_CONTEXT_SECONDS <= float(r['timestamp']) <= end_time
                    and (float(r['timestamp']) < end_time or int(r['index']) <= end_index)
                    and r['dst_id']==target_id and r['protocol']==protocol and r['dst_port_bucket']==service]
                packet_meta, flow_context = flow_features(w,context_rows)
                item['packet_meta']=np.asarray(packet_meta,dtype=np.float32)
                item['flow_context']=np.asarray(flow_context,dtype=np.float32)
            windows.append(item)
    # Preserve the paper's maximum of 5,000 sequences per class in total, split 8:1:1.
    caps = {'train':int(SAMPLES_PER_CLASS*TRAIN_RATIO),'val':int(SAMPLES_PER_CLASS*VAL_RATIO),'test':max(1,SAMPLES_PER_CLASS-int(SAMPLES_PER_CLASS*TRAIN_RATIO)-int(SAMPLES_PER_CLASS*VAL_RATIO))}
    rng = np.random.default_rng(RANDOM_SEED)
    selected = []
    for split in ('train','val','test'):
        for label in labels:
            subset = [w for w in windows if w['split']==split and w['label']==label]
            cap = max(1,caps[split])
            if len(subset)>cap:
                inds = np.sort(rng.choice(len(subset),cap,replace=False))
                subset = [subset[i] for i in inds]
            selected.extend(subset)
    print(f'{variant} windows by split:', {s:sum(w["split"]==s for w in selected) for s in ('train','val','test')})
    return selected, labels

def _dos_target(label):
    key = ''.join(ch for ch in label.lower() if ch.isalnum())
    return 1 if key.startswith('ddos') else 0 if key.startswith('dos') else -1

class WindowDataset(Dataset):
    def __init__(self, windows, split, variant):
        self.rows = [w for w in windows if w['split']==split]
        self.variant = variant
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        w = self.rows[idx]
        out = {
            'teacher_input_ids':torch.as_tensor(w['teacher_input_ids'],dtype=torch.long),
            'teacher_attention_mask':torch.as_tensor(w['teacher_attention_mask'],dtype=torch.long),
            'teacher_segment_ids':torch.as_tensor(w['teacher_segment_ids'],dtype=torch.long),
            'label':torch.tensor(w['label_id'],dtype=torch.long),
            'dos_target':torch.tensor(_dos_target(w['label']),dtype=torch.long),
        }
        if self.variant=='paper_baseline':
            out['input_ids']=torch.as_tensor(w['input_ids'],dtype=torch.long)
            out['attention_mask']=torch.as_tensor(w['attention_mask'],dtype=torch.long)
            out['segment_ids']=torch.as_tensor(w['teacher_segment_ids'],dtype=torch.long)
        else:
            out['packet_ids']=torch.as_tensor(w['packet_ids'],dtype=torch.long)
            out['packet_masks']=torch.as_tensor(w['packet_masks'],dtype=torch.long)
            out['packet_segments']=torch.as_tensor(w['packet_segments'],dtype=torch.long)
            out['packet_meta']=torch.as_tensor(w['packet_meta'],dtype=torch.float32)
            out['flow_context']=torch.as_tensor(w['flow_context'],dtype=torch.float32)
        return out

all_windows, class_names = build_windows(split_records, MODEL_VARIANT)
train_dataset = WindowDataset(all_windows,'train',MODEL_VARIANT)
val_dataset = WindowDataset(all_windows,'val',MODEL_VARIANT)
test_dataset = WindowDataset(all_windows,'test',MODEL_VARIANT)
if min(len(train_dataset),len(val_dataset),len(test_dataset)) == 0:
    raise ValueError('A split is empty. Check capture counts and window size.')


paper_baseline windows by split: {'train': 1600, 'val': 200, 'test': 200}


## 5. Shared teacher pretraining model


In [ ]:
class ModelConfig:
    def __init__(self,layers,hidden,heads,ffn,dropout=0.1):
        self.num_layers,self.hidden_size,self.num_heads,self.ffn_size=layers,hidden,heads,ffn
        self.vocab_size,self.max_seq_len,self.dropout=VOCAB_SIZE,MAX_SEQ_LEN,dropout

class TransformerBlock(nn.Module):
    def __init__(self,hidden,heads,ffn,dropout=0.1):
        super().__init__()
        self.attn=nn.MultiheadAttention(hidden,heads,dropout=dropout,batch_first=True)
        self.norm1=nn.LayerNorm(hidden)
        self.ffn=nn.Sequential(nn.Linear(hidden,ffn),nn.GELU(),nn.Linear(ffn,hidden),nn.Dropout(dropout))
        self.norm2=nn.LayerNorm(hidden); self.drop=nn.Dropout(dropout)
    def forward(self,x,attention_mask):
        a,weights=self.attn(x,x,x,key_padding_mask=~attention_mask.bool(),need_weights=True,average_attn_weights=False)
        x=self.norm1(x+self.drop(a)); x=self.norm2(x+self.drop(self.ffn(x)))
        return x,weights

class BERTBackbone(nn.Module):
    def __init__(self,cfg):
        super().__init__(); self.config=cfg
        self.token_embeddings=nn.Embedding(cfg.vocab_size,cfg.hidden_size,padding_idx=PAD_ID)
        self.position_embeddings=nn.Embedding(cfg.max_seq_len,cfg.hidden_size)
        self.segment_embeddings=nn.Embedding(2,cfg.hidden_size)
        self.embedding_norm=nn.LayerNorm(cfg.hidden_size); self.embedding_drop=nn.Dropout(cfg.dropout)
        self.layers=nn.ModuleList([TransformerBlock(cfg.hidden_size,cfg.num_heads,cfg.ffn_size,cfg.dropout) for _ in range(cfg.num_layers)])
    def forward(self,input_ids,attention_mask,segment_ids=None):
        if segment_ids is None: segment_ids=torch.zeros_like(input_ids)
        pos=torch.arange(input_ids.size(1),device=input_ids.device).unsqueeze(0)
        x=self.token_embeddings(input_ids)+self.position_embeddings(pos)+self.segment_embeddings(segment_ids)
        x=self.embedding_drop(self.embedding_norm(x)); hidden=[x]; attentions=[]
        for layer in self.layers:
            x,attn=layer(x,attention_mask); hidden.append(x); attentions.append(attn)
        return {'last_hidden_state':x,'cls_output':x[:,0],'all_hidden_states':hidden,'all_attention_weights':attentions}

class TeacherForPretraining(nn.Module):
    def __init__(self):
        super().__init__()
        self.config=ModelConfig(TEACHER_LAYERS,TEACHER_HIDDEN,TEACHER_HEADS,TEACHER_FFN)
        self.bert=BERTBackbone(self.config)
        self.mpm_dense=nn.Linear(TEACHER_HIDDEN,TEACHER_HIDDEN); self.mpm_norm=nn.LayerNorm(TEACHER_HIDDEN)
        self.mpm_decoder=nn.Linear(TEACHER_HIDDEN,VOCAB_SIZE); self.mpm_decoder.weight=self.bert.token_embeddings.weight
        self.spp_head=nn.Linear(TEACHER_HIDDEN,2)
    def forward(self,input_ids,attention_mask,mlm_labels=None,spp_labels=None,segment_ids=None):
        out=self.bert(input_ids,attention_mask,segment_ids)
        mpm=self.mpm_decoder(self.mpm_norm(F.gelu(self.mpm_dense(out['last_hidden_state']))))
        spp=self.spp_head(out['cls_output'])
        mlm_loss=None if mlm_labels is None else F.cross_entropy(mpm.reshape(-1,VOCAB_SIZE),mlm_labels.reshape(-1),ignore_index=-100)
        spp_loss=None if spp_labels is None else F.cross_entropy(spp,spp_labels)
        total=None if mlm_loss is None and spp_loss is None else (mlm_loss if mlm_loss is not None else 0)+(spp_loss if spp_loss is not None else 0)
        return {'total_loss':total,'mpm_loss':mlm_loss,'spp_loss':spp_loss,**out}

class PacketPretrainDataset(Dataset):
    def __init__(self,rows):
        self.rows=[r for r in rows if r['split']=='train']
        if len(self.rows)<2: raise ValueError('Need two training packets for SPP negatives.')
    def __len__(self): return len(self.rows)
    def __getitem__(self,idx):
        row=self.rows[idx]
        same=random.random()<0.5
        if same: payload=row['payload']; spp=1
        else:
            other=random.randrange(len(self.rows)-1)
            if other>=idx: other+=1
            payload=self.rows[other]['payload']; spp=0
        raw, raw_segments=tokenizer.encode_packet_segments(row['header_paper'],payload)
        ids,mask,segments=tokenizer.pad_segments(raw,raw_segments,MAX_SEQ_LEN)
        masked=list(ids); labels=[-100]*MAX_SEQ_LEN
        eligible=[]
        for pos,token in enumerate(ids):
            if not mask[pos] or token in (PAD_ID,CLS_ID,SEP_ID,MASK_ID): continue
            eligible.append(pos)
            if random.random()<MASK_PROB:
                labels[pos]=token; draw=random.random()
                if draw<0.8: masked[pos]=MASK_ID
                elif draw<0.9: masked[pos]=random.randint(TOKEN_OFFSET,VOCAB_SIZE-1)
        if eligible and all(v == -100 for v in labels):
            pos=random.choice(eligible); labels[pos]=ids[pos]; masked[pos]=MASK_ID
        return {'masked_input_ids':torch.tensor(masked,dtype=torch.long),'attention_mask':torch.tensor(mask,dtype=torch.long),
                'segment_ids':torch.tensor(segments,dtype=torch.long),'mlm_labels':torch.tensor(labels,dtype=torch.long),
                'spp_labels':torch.tensor(spp,dtype=torch.long)}

def parameter_count(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)
def autocast_context(): return torch.cuda.amp.autocast(enabled=(DEVICE=='cuda'))
def make_scaler(): return torch.cuda.amp.GradScaler(enabled=(DEVICE=='cuda'))
def move_batch(batch): return {k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}

def save_teacher_checkpoint(model,optimizer,scheduler,step):
    torch.save({'model_state_dict':model.state_dict(),'optimizer_state_dict':optimizer.state_dict(),
        'scheduler_state_dict':scheduler.state_dict(),'step':step,'dataset_fingerprint':DATASET_FINGERPRINT,
        'pretrain_steps':PRETRAIN_STEPS,'seed':RANDOM_SEED},SHARED_TEACHER_PATH)

def load_or_train_teacher():
    teacher=TeacherForPretraining().to(DEVICE)
    ckpt=None
    if os.path.exists(SHARED_TEACHER_PATH):
        ckpt=torch.load(SHARED_TEACHER_PATH,map_location=DEVICE,weights_only=False)
        if ckpt.get('dataset_fingerprint')!=DATASET_FINGERPRINT:
            raise ValueError('Shared teacher belongs to another PCAP set. Use a new EXPERIMENT_ROOT or remove the stale checkpoint.')
        if int(ckpt.get('pretrain_steps',0))==PRETRAIN_STEPS and int(ckpt.get('step',0))>=PRETRAIN_STEPS:
            teacher.load_state_dict(ckpt['model_state_dict']); teacher.eval()
            print(f'Reusing shared teacher at step {ckpt["step"]:,}.'); return teacher
    dataset=PacketPretrainDataset(split_records)
    loader=DataLoader(dataset,batch_size=MICRO_BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=(DEVICE=='cuda'),drop_last=False,generator=torch.Generator().manual_seed(RANDOM_SEED))
    optimizer=AdamW(teacher.parameters(),lr=PRETRAIN_LR)
    total=PRETRAIN_STEPS; warm=int(total*WARMUP_RATIO)
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda s:min(1.0,s/max(1,warm)) if s<warm else max(0.0,(total-s)/max(1,total-warm)))
    start=0
    if ckpt is not None:
        if ckpt.get('pretrain_steps')==PRETRAIN_STEPS:
            teacher.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict']); scheduler.load_state_dict(ckpt['scheduler_state_dict']); start=int(ckpt['step'])
        else:
            print('Checkpoint training budget differs; starting from fresh initialized weights for this config.')
    teacher.train(); it=iter(loader); scaler=make_scaler(); optimizer.zero_grad(set_to_none=True)
    pbar=tqdm(total=PRETRAIN_STEPS,initial=start,desc='Teacher MPM + SPP')
    for step in range(start,PRETRAIN_STEPS):
        for _ in range(GRAD_ACCUM_STEPS):
            try: raw_batch=next(it)
            except StopIteration: it=iter(loader); raw_batch=next(it)
            batch=move_batch(raw_batch)
            with autocast_context():
                out=teacher(batch['masked_input_ids'],batch['attention_mask'],batch['mlm_labels'],batch['spp_labels'],batch['segment_ids'])
                loss=out['total_loss']/GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(teacher.parameters(),1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        pbar.update(1); pbar.set_postfix(loss=f'{float(loss.item()*GRAD_ACCUM_STEPS):.4f}')
        if (step+1)%CHECKPOINT_EVERY==0: save_teacher_checkpoint(teacher,optimizer,scheduler,step+1)
    pbar.close(); save_teacher_checkpoint(teacher,optimizer,scheduler,PRETRAIN_STEPS); teacher.eval()
    return teacher

def masked_hidden_mse(s,t,mask):
    m=mask.unsqueeze(-1).to(s.dtype); d=(s-t)**2
    return (d*m).sum()/(m.sum()*d.size(-1)).clamp_min(1.0)

def masked_attention_mse(s,t,mask):
    m=mask[:,None,:,None]*mask[:,None,None,:]; d=(s-t)**2
    return (d*m.to(d.dtype)).sum()/(m.sum()*d.size(1)).clamp_min(1.0)

def compute_class_metrics(y,p,labels):
    pr,re,f,support=precision_recall_fscore_support(y,p,labels=list(range(len(labels))),zero_division=0)
    return {'accuracy':float(accuracy_score(y,p)),'macro_precision':float(np.mean(pr)),'macro_recall':float(np.mean(re)),
        'macro_f1':float(np.mean(f)),'weighted_f1':float(precision_recall_fscore_support(y,p,average='weighted',zero_division=0)[2]),
        'per_class':{labels[i]:{'precision':float(pr[i]),'recall':float(re[i]),'f1':float(f[i]),'support':int(support[i])} for i in range(len(labels))},
        'confusion_matrix':confusion_matrix(y,p,labels=list(range(len(labels)))).tolist()}

def plot_cm(metrics,labels,title):
    cm=np.array(metrics['confusion_matrix'])
    plt.figure(figsize=(max(8,len(labels)*0.8),max(6,len(labels)*0.65)))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=labels,yticklabels=labels)
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(title); plt.tight_layout(); plt.show()


## 6. Student architecture


In [ ]:
PAPER_CONFIG=ModelConfig(STUDENT_LAYERS,STUDENT_HIDDEN,STUDENT_HEADS,STUDENT_FFN,dropout=0.1)
class PaperStudentClassifier(nn.Module):
    def __init__(self,n_classes):
        super().__init__()
        self.bert=BERTBackbone(PAPER_CONFIG); self.dropout=nn.Dropout(0.5)
        self.classifier=nn.Linear(STUDENT_HIDDEN,n_classes)
    def forward(self,input_ids,attention_mask,segment_ids=None):
        out=self.bert(input_ids,attention_mask,segment_ids)
        return {'z':out['cls_output'],'logits':self.classifier(self.dropout(out['cls_output']))}


## 7. Distillation, fine-tuning, and evaluation


In [ ]:
def model_for_batch(model,batch,variant):
    if variant=='paper_baseline': return model(batch['input_ids'],batch['attention_mask'],batch['segment_ids'])
    return model(batch['packet_ids'],batch['packet_masks'],batch['packet_meta'],batch['flow_context'],batch['packet_segments'])

def make_student(variant,n_classes):
    return PaperStudentClassifier(n_classes) if variant=='paper_baseline' else FlowAwareStudent(n_classes)

def make_distill_loader(windows,variant):
    ds=WindowDataset(windows,'train',variant)
    return DataLoader(ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=(DEVICE=='cuda'),generator=torch.Generator().manual_seed(RANDOM_SEED))

def distill_paper(teacher,student,loader):
    teacher.eval(); student.train()
    pa=nn.ModuleList([nn.Linear(STUDENT_HEADS,TEACHER_HEADS,bias=False) for _ in range(STUDENT_LAYERS)]).to(DEVICE)
    ph=nn.ModuleList([nn.Linear(STUDENT_HIDDEN,TEACHER_HIDDEN,bias=False) for _ in range(STUDENT_LAYERS)]).to(DEVICE)
    params=list(student.parameters())+list(pa.parameters())+list(ph.parameters())
    opt=AdamW(params,lr=DISTILL_LR); scaler=make_scaler()
    for epoch in range(DISTILL_EPOCHS):
        opt.zero_grad(set_to_none=True); running=0.0
        pbar=tqdm(loader,desc=f'Paper distillation {epoch+1}/{DISTILL_EPOCHS}')
        for i,raw in enumerate(pbar):
            b=move_batch(raw)
            with torch.no_grad(),autocast_context(): t=teacher.bert(b['teacher_input_ids'],b['teacher_attention_mask'],b['teacher_segment_ids'])
            with autocast_context():
                s=student.bert(b['input_ids'],b['attention_mask'],b['segment_ids'])
                la=torch.zeros((),device=DEVICE); lh=torch.zeros((),device=DEVICE)
                for si,ti in [(0,5),(1,11)]:
                    sa=s['all_attention_weights'][si]; ta=t['all_attention_weights'][ti]
                    batch,heads,q,k=sa.shape
                    sa=sa.permute(0,2,3,1).reshape(-1,heads)
                    sa=pa[si](sa).reshape(batch,q,k,TEACHER_HEADS).permute(0,3,1,2)
                    la=la+masked_attention_mse(sa.float(),ta.float(),b['teacher_attention_mask'])
                    sh=ph[si](s['all_hidden_states'][si+1])
                    lh=lh+masked_hidden_mse(sh.float(),t['all_hidden_states'][ti+1].float(),b['teacher_attention_mask'])
                loss=la+lh
            scaler.scale(loss/GRAD_ACCUM_STEPS).backward()
            if (i+1)%GRAD_ACCUM_STEPS==0 or i+1==len(loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params,1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            running+=float(loss.detach().item()); pbar.set_postfix(loss=f'{running/(i+1):.4f}')
    return student

def distill_flow(teacher,student,loader):
    teacher.eval(); student.train()
    projection=nn.Linear(STUDENT_HIDDEN,TEACHER_HIDDEN,bias=False).to(DEVICE)
    params=list(student.parameters())+list(projection.parameters())
    opt=AdamW(params,lr=DISTILL_LR); scaler=make_scaler()
    for epoch in range(DISTILL_EPOCHS):
        opt.zero_grad(set_to_none=True); running=0.0
        pbar=tqdm(loader,desc=f'Flow distillation {epoch+1}/{DISTILL_EPOCHS}')
        for i,raw in enumerate(pbar):
            b=move_batch(raw)
            with torch.no_grad(),autocast_context(): t=teacher.bert(b['teacher_input_ids'],b['teacher_attention_mask'],b['teacher_segment_ids'])
            with autocast_context():
                s=student(b['packet_ids'],b['packet_masks'],b['packet_meta'],b['flow_context'],b['packet_segments'])
                loss=F.mse_loss(projection(s['z']).float(),t['cls_output'].float())
            scaler.scale(loss/GRAD_ACCUM_STEPS).backward()
            if (i+1)%GRAD_ACCUM_STEPS==0 or i+1==len(loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params,1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            running+=float(loss.detach().item()); pbar.set_postfix(loss=f'{running/(i+1):.4f}')
    return student

def evaluate_classifier(model,loader,variant,labels):
    model.eval(); y,p,losses=[],[],[]
    with torch.no_grad():
        for raw in loader:
            b=move_batch(raw); out=model_for_batch(model,b,variant)
            losses.append(float(F.cross_entropy(out['logits'],b['label']).item()))
            y.extend(b['label'].cpu().tolist()); p.extend(out['logits'].argmax(-1).cpu().tolist())
    m=compute_class_metrics(y,p,labels); m['loss']=float(np.mean(losses)) if losses else 0.0
    return m

def fit_classifier(model,train_loader,val_loader,variant,labels):
    model.to(DEVICE)
    dos_label_targets=[_dos_target(name) for name in labels]
    use_dos_aux=(0 in dos_label_targets and 1 in dos_label_targets)
    opt=AdamW(model.parameters(),lr=FINETUNE_LR)
    updates=max(1,math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)*FINETUNE_EPOCHS); warm=int(updates*WARMUP_RATIO)
    sched=torch.optim.lr_scheduler.LambdaLR(opt,lambda s:min(1.0,s/max(1,warm)) if s<warm else max(0.0,(updates-s)/max(1,updates-warm)))
    scaler=make_scaler(); best=-1.0; best_state=None; history=[]
    for epoch in range(FINETUNE_EPOCHS):
        model.train(); opt.zero_grad(set_to_none=True); running=0.0
        pbar=tqdm(train_loader,desc=f'Fine-tuning {epoch+1}/{FINETUNE_EPOCHS}')
        for i,raw in enumerate(pbar):
            b=move_batch(raw)
            with autocast_context():
                out=model_for_batch(model,b,variant)
                loss=F.cross_entropy(out['logits'],b['label'])
                if variant=='flow_aware' and use_dos_aux:
                    mask=b['dos_target']>=0
                    if mask.any(): loss=loss+DOS_AUX_WEIGHT*F.cross_entropy(out['dos_logits'][mask],b['dos_target'][mask])
            scaler.scale(loss/GRAD_ACCUM_STEPS).backward()
            if (i+1)%GRAD_ACCUM_STEPS==0 or i+1==len(train_loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            running+=float(loss.detach().item()); pbar.set_postfix(loss=f'{running/(i+1):.4f}')
        val=evaluate_classifier(model,val_loader,variant,labels)
        history.append({'epoch':epoch+1,'train_loss':running/max(1,len(train_loader)),'val_loss':val['loss'],'val_macro_f1':val['macro_f1']})
        print(f'Validation macro-F1: {val["macro_f1"]:.4f}')
        if val['macro_f1']>best:
            best=val['macro_f1']; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            torch.save(best_state,BEST_MODEL_PATH)
    if best_state is not None: model.load_state_dict(best_state)
    return model,history

def measure_latency_ms(model,dataset,variant,device,runs=100):
    sample=dataset[0]
    batch={k:v.unsqueeze(0).to(device) for k,v in sample.items()}
    original_device=next(model.parameters()).device
    model.to(device); model.eval()
    def sync():
        if str(device).startswith('cuda'):
            torch.cuda.synchronize(device)
    with torch.inference_mode():
        for _ in range(20):
            model_for_batch(model,batch,variant)
        sync()
        times=[]
        for _ in range(runs):
            sync(); start=time.perf_counter()
            model_for_batch(model,batch,variant)
            sync(); times.append((time.perf_counter()-start)*1000.0)
    model.to(original_device)
    return {'median':float(np.median(times)),'p95':float(np.percentile(times,95)),'runs':runs,'warmup_runs':20,'batch_size':1}

def run_experiment(variant):
    windows,labels=build_windows(split_records,variant)
    train_ds,val_ds,test_ds=(WindowDataset(windows,s,variant) for s in ('train','val','test'))
    train_loader=DataLoader(train_ds,batch_size=MICRO_BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=(DEVICE=='cuda'),generator=torch.Generator().manual_seed(RANDOM_SEED))
    val_loader=DataLoader(val_ds,batch_size=MICRO_BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=(DEVICE=='cuda'),generator=torch.Generator().manual_seed(RANDOM_SEED))
    test_loader=DataLoader(test_ds,batch_size=MICRO_BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=(DEVICE=='cuda'),generator=torch.Generator().manual_seed(RANDOM_SEED))
    teacher=load_or_train_teacher()
    seed_everything(RANDOM_SEED+1)
    student=make_student(variant,len(labels)).to(DEVICE)
    distill_loader=make_distill_loader(windows,variant)
    if variant=='paper_baseline': student.bert=distill_paper(teacher,student.bert,distill_loader)
    else: student=distill_flow(teacher,student,distill_loader)
    del teacher
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    model,history=fit_classifier(student,train_loader,val_loader,variant,labels)
    test=evaluate_classifier(model,test_loader,variant,labels)
    plot_cm(test,labels,f'{variant} test confusion matrix')
    latency_cpu=measure_latency_ms(model,test_ds,variant,'cpu')
    latency_gpu=measure_latency_ms(model,test_ds,variant,DEVICE) if DEVICE=='cuda' else None
    result={
        'variant':variant,'dataset':DATASET_NAME,'dataset_fingerprint':DATASET_FINGERPRINT,'split_manifest':split_manifest,
        'class_names':labels,'config':{'seed':RANDOM_SEED,'pretrain_steps':PRETRAIN_STEPS,'effective_batch':MICRO_BATCH_SIZE*GRAD_ACCUM_STEPS,
        'pretrain_lr':PRETRAIN_LR,'distill_lr':DISTILL_LR,'finetune_lr':FINETUNE_LR,'distill_epochs':DISTILL_EPOCHS,
        'finetune_epochs':FINETUNE_EPOCHS,'window_size':WINDOW_SIZE,'window_stride':WINDOW_STRIDE,'flow_context_seconds':FLOW_CONTEXT_SECONDS,'samples_per_class':SAMPLES_PER_CLASS,
        'pretrain_batch_size':PRETRAIN_BATCH_SIZE,'distill_batch_size':DISTILL_BATCH_SIZE,'finetune_batch_size':FINETUNE_BATCH_SIZE,
        'warmup_ratio':WARMUP_RATIO,'mask_probability':MASK_PROB,'dos_aux_weight':DOS_AUX_WEIGHT,
        'dos_aux_active':(0 in [_dos_target(name) for name in labels] and 1 in [_dos_target(name) for name in labels]),
        'flow_token_embedding_dim':32,'classifier_dropout':0.5,
        'student_layers':STUDENT_LAYERS,'student_hidden':STUDENT_HIDDEN,'student_heads':STUDENT_HEADS,'student_ffn':STUDENT_FFN,'quick_run':QUICK_RUN},
        'train_windows':len(train_ds),'validation_windows':len(val_ds),'test_windows':len(test_ds),
        'student_parameters':parameter_count(model),'latency_ms_batch1':{'cpu':latency_cpu,'gpu':latency_gpu},
        'validation_history':history,'test_metrics':test}
    with open(RESULTS_PATH,'w') as f: json.dump(result,f,indent=2)
    torch.save(model.state_dict(),BEST_MODEL_PATH)
    print('Saved:',RESULTS_PATH)
    print('Batch-one latency (ms):',result['latency_ms_batch1'])
    print(json.dumps({k:test[k] for k in ('accuracy','macro_precision','macro_recall','macro_f1','weighted_f1')},indent=2))
    return result

def show_dos_ddos_confusion(result):
    labels=result['class_names']; keys=[''.join(c for c in n.lower() if c.isalnum()) for n in labels]
    idx=[i for i,k in enumerate(keys) if k.startswith('dos') or k.startswith('ddos')]
    if len(idx)<2: print('Dataset does not contain at least two DoS/DDoS classes.'); return
    cm=np.array(result['test_metrics']['confusion_matrix'])
    print('DoS/DDoS confusion submatrix (true rows, predicted columns):')
    print([labels[i] for i in idx]); print(cm[np.ix_(idx,idx)])


## 8. Run the experiment


In [ ]:
MODEL_VARIANT='paper_baseline'
VARIANT_DIR=os.path.join(EXPERIMENT_ROOT,DATASET_NAME,f'seed_{RANDOM_SEED}',MODEL_VARIANT); os.makedirs(VARIANT_DIR,exist_ok=True)
RESULTS_PATH=os.path.join(VARIANT_DIR,'metrics.json'); BEST_MODEL_PATH=os.path.join(VARIANT_DIR,'best_student.pt')
result=run_experiment(MODEL_VARIANT)
show_dos_ddos_confusion(result)


paper_baseline windows by split: {'train': 1600, 'val': 200, 'test': 200}


Teacher MPM + SPP:   0%|          | 0/1000 [00:00<?, ?it/s]